In [1]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from func.UsefulFunctions import prettifyGenres

This notebook takes the group data (that was extracted from the Music Brainz JSON) and inspects it. Groups are groups, choirs, orchestras unless stated otherwise.

In [3]:
groupDF = pl.read_parquet("/home/aja25/Documents/fromCluster/ensemblesMusicBrainzData.parquet")

#drop the relations table as it is no longer needed
groupDF = groupDF.drop("relations")
#De-duplicate the data
groupDF = groupDF.unique()

num_of_ids = groupDF.select(
    pl.col("id")
).unique().count().item(0,0)
print(f"There are {num_of_ids} unique IDs in the dataset (that is a snapshot from 18/07/2026)")


shape: (1, 1)
┌────────┐
│ id     │
│ ---    │
│ u32    │
╞════════╡
│ 672717 │
└────────┘


In [4]:
#How many people do not have a name in this dataset?
num_people_with_no_names = groupDF.select(
    pl.col("name"),
    pl.col("id")
).unique().null_count()
print(num_people_with_no_names)
print(f"There are {num_people_with_no_names.item(0,0)} entities with no name, and {num_people_with_no_names.item(0,1)} entities with no ids.")

shape: (1, 2)
┌──────┬─────┐
│ name ┆ id  │
│ ---  ┆ --- │
│ u32  ┆ u32 │
╞══════╪═════╡
│ 0    ┆ 0   │
└──────┴─────┘
There are 0 entities with no name, and 0 entities with no ids.


In [5]:
#check if there are any duplicates in ids
ids = groupDF.select(
    pl.col("id")
).is_duplicated()
print(ids.value_counts())

print("Based off of the lack of trues in the is_duplicated function. All rows are unique after the deduplication ")

shape: (1, 2)
┌───────┬────────┐
│       ┆ count  │
│ ---   ┆ ---    │
│ bool  ┆ u32    │
╞═══════╪════════╡
│ false ┆ 672717 │
└───────┴────────┘
Based off of the lack of trues in the is_duplicated function. All rows are unique after the deduplication 


In [7]:
#Do a count for the countries for the dataset.
countryCount = groupDF.select(pl.col("country").value_counts(sort=True)).unnest("country")
print(countryCount)

shape: (236, 2)
┌─────────┬────────┐
│ country ┆ count  │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ null    ┆ 203557 │
│ US      ┆ 121124 │
│ GB      ┆ 45511  │
│ DE      ┆ 33522  │
│ JP      ┆ 29454  │
│ …       ┆ …      │
│ BQ      ┆ 1      │
│ SS      ┆ 1      │
│ AN      ┆ 1      │
│ MS      ┆ 1      │
│ WF      ┆ 1      │
└─────────┴────────┘


Based off the table, the most common countries are (outside of null which shows that most groups do not have a country attached to them is) US, GB, DE, and japan

In [9]:
#What is the most common type of group for the entities
typeCount = groupDF.group_by("type").agg(pl.col("id").count())
print(typeCount)

type,id
str,u32
"""Orchestra""",11760
"""Choir""",10771
"""Group""",650186
